<h1 dir="rtl" style="text-align: right;">
تحلیل اکتشافی داده های اضطراب اجتماعی
</h1>

<p dir="rtl" style="text-align: right;">
<strong>فاز دوم: <span dir="ltr">EDA</span></strong>
</p>

<p dir="rtl" style="text-align: right;">
اعضای تیم: علی خوش اخلاق، محمدحسین میرمعصومی، آرمین نورمحمدی، علی کریمی، محسن منصف
</p>

<h2 dir="rtl" style="text-align: right;">
1. کتابخانه ها و تنظیمات
</h2>

<p dir="rtl" style="text-align: right;">
از پایتون 3.13 استفاده کنید و ورژن های زیر
</p>

In [ ]:
# pandas==3.0.5 numpy==2.2.6 plotly==7.1.0 scipy==1.16.3 statsmodels==0.15.0 matplotlib==3.11.2

In [ ]:
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt
import scipy
import statsmodels
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pio.renderers.default = "notebook"
px.defaults.template = "plotly_dark"
px.defaults.height = 450
BLUE = "#4C78A8"
RED = "#E45756"
TARGET = "Anxiety Level (1-10)"
LABEL = "Target"

<h2 dir="rtl" style="text-align: right;">
2. داده های اولیه
</h2>

In [ ]:
df = pd.read_csv("social_anxiety_dataset.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
pd.DataFrame({
    'Missing': df.isnull().sum(),
    'Missing_%': (df.isna().mean() * 100).round(1),
    'unique': df.nunique(),
})

In [ ]:
print(df.duplicated().sum())

<h2 dir="rtl" style="text-align: right;">
مشاهدات اولیه
</h2>

<p dir="rtl" style="text-align: right;">
• سطر هامون 2030 تاس و ستون هامون 22 تاس
</p>

<p dir="rtl" style="text-align: right;">
• 9 تا ستون با داده گمشده داریم
</p>

<p dir="rtl" style="text-align: right;">
&nbsp;&nbsp;&nbsp;&nbsp;◦ که <span dir="ltr">Therapy History</span> با 90% داده گمشده بدترینه
</p>

<p dir="rtl" style="text-align: right;">
• <span dir="ltr">Sleep Hours</span>، <span dir="ltr">Physical Activity</span> و <span dir="ltr">Alcohol</span> مقدار منفی دارند
</p>

<p dir="rtl" style="text-align: right;">
• <span dir="ltr">Stress Level</span> مقدار 15 دارد که منطقی نیست
</p>

<p dir="rtl" style="text-align: right;">
• دوتا ستون <span dir="ltr">Target</span> و <span dir="ltr">is_Anxious</span> تغریبا یکسانند
</p>

<p dir="rtl" style="text-align: right;">
• ستون <span dir="ltr">Heart Rate</span> و <span dir="ltr">Caffeine</span> مقدار خارج از بازه دارند
</p>

<h2 dir="rtl" style="text-align: right;">
3. تحلیل و پاک سازی داده
</h2>

<h3 dir="rtl" style="text-align: right;">
3.1 جمعیت شناختی:
<span dir="ltr">Age, Gender, Occupation</span>
</h3>

<h4 dir="rtl" style="text-align: right;">
انتخاب متغیرهای جمعیت‌شناختی
</h4>

<p dir="rtl" style="text-align: right;">
در این بخش، سه متغیر
<span dir="ltr">Age</span>،
<span dir="ltr">Gender</span>
و
<span dir="ltr">Occupation</span>
به‌عنوان متغیرهای جمعیت‌شناختی انتخاب شدند.
برای بررسی و پاک‌سازی این متغیرها، یک کپی مستقل از این سه ستون ساخته می‌شود تا تغییرات این بخش به‌صورت کنترل‌شده انجام شوند.
</p>

In [ ]:
demographic_df = df[
    [
        "Age",
        "Gender",
        "Occupation"
    ]
].copy()

demographic_df.head()

<h4 dir="rtl" style="text-align: right;">
بررسی اولیه متغیرهای جمعیت‌شناختی
</h4>

<p dir="rtl" style="text-align: right;">
پیش از انجام پاک‌سازی، نوع داده، تعداد مقادیر گمشده و تعداد مقادیر یکتای سه متغیر جمعیت‌شناختی بررسی می‌شود.
این مرحله کمک می‌کند مشکلات موجود در داده قبل از هرگونه تغییر شناسایی شوند.
</p>

In [ ]:
pd.DataFrame({
    "Data Type": demographic_df.dtypes,
    "Missing": demographic_df.isna().sum(),
    "Missing %": (demographic_df.isna().mean() * 100).round(1),
    "Unique": demographic_df.nunique()
})

<p dir="rtl" style="text-align: right;">
نتایج بررسی اولیه نشان داد که ستون
<span dir="ltr">Age</span>
دارای ۶۲ مقدار گمشده، معادل حدود ۳.۱ درصد داده‌ها است.
ستون
<span dir="ltr">Gender</span>
نیز دارای ۱۱۹ مقدار گمشده، معادل حدود ۵.۹ درصد داده‌ها است.
</p>

<p dir="rtl" style="text-align: right;">
در ستون
<span dir="ltr">Occupation</span>
هیچ مقدار گمشده‌ای مشاهده نشد.
همچنین این ستون شامل ۱۳ مقدار یکتا است.
بنابراین، در ادامه لازم است ستون‌های
<span dir="ltr">Age</span>
و
<span dir="ltr">Gender</span>
با دقت بیشتری بررسی شوند.
</p>

<h4 dir="rtl" style="text-align: right;">
بررسی متغیر سن
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Age</span>
یک متغیر عددی است.
برای بررسی دامنه، مرکز و پراکندگی مقادیر آن، از متد
<span dir="ltr">describe()</span>
استفاده می‌شود.
همچنین دامنه مقادیر برای شناسایی سن‌های نامعتبر یا غیرعادی بررسی می‌شود.
</p>

In [ ]:
demographic_df["Age"].describe()

<p dir="rtl" style="text-align: right;">
آمار توصیفی نشان داد که سن افراد بین ۱۸ تا ۶۴ سال قرار دارد.
میانگین سن حدود ۳۹.۹ سال و میانه برابر با ۴۰ سال است.
بنابراین، در دامنه سن مقادیر نامعتبر واضح مانند سن منفی یا بسیار بزرگ مشاهده نشد.
</p>

<p dir="rtl" style="text-align: right;">
از آنجا که ۶۲ مقدار در ستون
<span dir="ltr">Age</span>
گمشده است، برای حفظ این ردیف‌ها مقادیر گمشده با میانه سن جایگزین می‌شوند.
استفاده از میانه باعث می‌شود مقدار مرکزی داده حفظ شود و نسبت به مقادیر بسیار کوچک یا بزرگ حساسیت کمتری داشته باشد.
</p>

In [ ]:
age_median = demographic_df["Age"].median()

demographic_df["Age"] = (
    demographic_df["Age"]
    .fillna(age_median)
    .astype(int)
)

print("Median Age:", age_median)
print("Missing Age:", demographic_df["Age"].isna().sum())

<h4 dir="rtl" style="text-align: right;">
بررسی متغیر جنسیت
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Gender</span>
یک متغیر دسته‌ای است.
برای بررسی دسته‌های موجود، فراوانی هر دسته و مقادیر گمشده، از متد
<span dir="ltr">value_counts()</span>
استفاده می‌شود.
</p>

In [ ]:
demographic_df["Gender"].value_counts(dropna=False)

<p dir="rtl" style="text-align: right;">
نتایج نشان داد که مقادیر ثبت‌شده در ستون
<span dir="ltr">Gender</span>
فقط شامل سه دسته معتبر
<span dir="ltr">Female</span>،
<span dir="ltr">Male</span>
و
<span dir="ltr">Other</span>
هستند و ناسازگاری در نام‌گذاری دسته‌ها مشاهده نشد.
</p>

<p dir="rtl" style="text-align: right;">
با این حال، ۱۱۹ مقدار گمشده در این ستون وجود دارد.
از آنجا که اطلاعات کافی برای تعیین جنسیت واقعی این افراد وجود ندارد، جایگزینی آن‌ها با پرتکرارترین دسته می‌تواند توزیع داده را به‌صورت مصنوعی تغییر دهد.
بنابراین، این مقادیر با برچسب
<span dir="ltr">Unknown</span>
مشخص می‌شوند.
</p>

In [ ]:
demographic_df["Gender"] = (
    demographic_df["Gender"]
    .fillna("Unknown")
)

demographic_df["Gender"].value_counts(dropna=False)

<h4 dir="rtl" style="text-align: right;">
بررسی متغیر شغل
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Occupation</span>
یک متغیر دسته‌ای است.
برای بررسی دسته‌های موجود، فراوانی هر شغل و وجود مقادیر گمشده، فراوانی مقادیر این ستون بررسی می‌شود.
</p>

In [ ]:
demographic_df["Occupation"].value_counts(dropna=False)

<p dir="rtl" style="text-align: right;">
ستون
<span dir="ltr">Occupation</span>
شامل ۱۳ دسته شغلی است و هیچ مقدار گمشده‌ای در آن وجود ندارد.
همچنین در بررسی دسته‌های موجود، ناسازگاری مشخصی در نام‌گذاری مقادیر مشاهده نشد.
بنابراین، این ستون بدون تغییر باقی می‌ماند.
</p>

<h4 dir="rtl" style="text-align: right;">
بررسی نهایی پاک‌سازی داده‌های جمعیت‌شناختی
</h4>

<p dir="rtl" style="text-align: right;">
پس از انجام پاک‌سازی، وضعیت نهایی سه متغیر جمعیت‌شناختی دوباره بررسی می‌شود تا از حذف مقادیر گمشده و صحیح بودن نوع داده‌ها اطمینان حاصل شود.
</p>

In [ ]:
pd.DataFrame({
    "Missing": demographic_df.isna().sum(),
    "Unique": demographic_df.nunique(),
    "Data Type": demographic_df.dtypes
})

<h4 dir="rtl" style="text-align: right;">
جمع‌بندی پاک‌سازی متغیرهای جمعیت‌شناختی
</h4>

<p dir="rtl" style="text-align: right;">
در ستون
<span dir="ltr">Age</span>
تعداد ۶۲ مقدار گمشده با میانه سن، برابر با ۴۰ سال، جایگزین شد و نوع داده این ستون به عدد صحیح تبدیل شد.
</p>

<p dir="rtl" style="text-align: right;">
در ستون
<span dir="ltr">Gender</span>
تعداد ۱۱۹ مقدار گمشده با برچسب
<span dir="ltr">Unknown</span>
مشخص شد تا بدون فرض کردن جنسیت افراد، اطلاعات این ردیف‌ها در دیتاست حفظ شود.
</p>

<p dir="rtl" style="text-align: right;">
ستون
<span dir="ltr">Occupation</span>
فاقد مقدار گمشده یا دسته نامعتبر بود و بدون تغییر باقی ماند.
در پایان، هیچ مقدار گمشده‌ای در سه متغیر جمعیت‌شناختی باقی نماند.
</p>

<h3 dir="rtl" style="text-align: right;">
3.2 سبک زندگی:
<span dir="ltr">Sleep Hours, Physical Activity, Caffeine Intake, Alcohol Consumption, Smoking, Diet Quality</span>
</h3>

In [ ]:
lifestyle_cols = [
    'Sleep Hours',
    'Physical Activity (hrs/week)',
    'Caffeine Intake (mg/day)',
    'Alcohol Consumption (drinks/week)',
    'Smoking',
    'Diet Quality (1-10)'
]

lifestyle_df = df[lifestyle_cols].copy()
#==================
# Numeric Data
#==================
Sleep_Hours = lifestyle_df['Sleep Hours']
Physical_Activity = lifestyle_df['Physical Activity (hrs/week)']
Caffeine_Intake = lifestyle_df['Caffeine Intake (mg/day)']
Alcohol_Consumption = lifestyle_df['Alcohol Consumption (drinks/week)']

# =========================
# Categorical Data
# =========================
Smoking = lifestyle_df['Smoking']
Diet_Quality = lifestyle_df['Diet Quality (1-10)']

In [ ]:
# =========================
# Plot Numerical Data
# =========================
fig, axes = plt.subplots(4, 1, figsize=(12, 22))

# --- 1. Sleep Hours ---
axes[0].hist(
    Sleep_Hours,
    bins=20,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[0].set_title('Distribution of Sleep Hours', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sleep Hours', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)


# --- 2. Physical Activity ---
axes[1].hist(
    Physical_Activity,
    bins=20,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[1].set_title('Distribution of Physical Activity', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Physical Activity (hrs/week)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)


# --- 3. Caffeine Intake ---
axes[2].hist(
    Caffeine_Intake,
    bins=20,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[2].set_title('Distribution of Caffeine Intake', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Caffeine Intake (mg/day)', fontsize=11)
axes[2].set_ylabel('Frequency', fontsize=11)
axes[2].grid(axis='y', alpha=0.3)


# --- 4. Alcohol Consumption ---
axes[3].hist(
    Alcohol_Consumption,
    bins=10,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[3].set_title('Distribution of Alcohol Consumption', fontsize=14, fontweight='bold')
axes[3].set_xlabel('Alcohol Consumption (drinks/week)', fontsize=11)
axes[3].set_ylabel('Frequency', fontsize=11)
axes[3].grid(axis='y', alpha=0.3)

# =========================
# Layout
# =========================
fig.suptitle(
    'Distribution of Numeric Variables',
    fontsize=18,
    fontweight='bold',
    y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
# =========================
# Plot categorical Data
# =========================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 1.Smoking ---
smoking_counts = Smoking.value_counts()

axes[0].bar(
    smoking_counts.index.astype(str),
    smoking_counts.values,
    edgecolor='black',
    alpha=0.8
)

axes[0].set_title('Smoking Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Smoking', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)


# --- 2.Diet Quality ---
diet_counts = Diet_Quality.value_counts().sort_index()

axes[1].bar(
    diet_counts.index.astype(str),
    diet_counts.values,
    edgecolor='black',
    alpha=0.8
)

axes[1].set_title('Diet Quality Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Diet Quality (1-10)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)


# =========================
# Layout
# =========================
fig.suptitle(
    'Distribution of Categorical and Discrete Variables',
    fontsize=17,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# remove invlid data from Numeric columns 
# =========================

lifestyle_df.loc[lifestyle_df['Sleep Hours'] < 0, 'Sleep Hours'] = np.nan

lifestyle_df.loc[lifestyle_df['Physical Activity (hrs/week)'] < 0,
       'Physical Activity (hrs/week)'] = np.nan

lifestyle_df.loc[lifestyle_df['Alcohol Consumption (drinks/week)'] < 0,
       'Alcohol Consumption (drinks/week)'] = np.nan

In [ ]:
numeric_cols = [
    'Sleep Hours',
    'Physical Activity (hrs/week)',
    'Caffeine Intake (mg/day)',
    'Alcohol Consumption (drinks/week)'
]

# Calculate median and replace missing values
for col in numeric_cols:
    median_value = lifestyle_df[col].median()
    lifestyle_df[col] = lifestyle_df[col].fillna(median_value)

In [ ]:
smoking_mode = Smoking.mode()[0]
print('Smoking Mode:', smoking_mode)
lifestyle_df['Smoking'] = lifestyle_df['Smoking'].fillna(smoking_mode)

Diet_Quality_mode = Diet_Quality.mode()[0]
print('Diet Quality:', Diet_Quality_mode)
lifestyle_df['Diet Quality (1-10)'] = lifestyle_df['Diet Quality (1-10)'].fillna(Diet_Quality_mode)

<h3 dir="rtl" style="text-align: right;">
3.3 فیزیولوژیک:
<span dir="ltr">Heart Rate, Breathing Rate, Sweating Level, Dizziness</span>
</h3>

<h4 dir="rtl" style="text-align: right;">
روش بررسی متغیرهای فیزیولوژیک
</h4>

<p dir="rtl" style="text-align: right;">
در این بخش، نوع داده، مقادیر گمشده، دامنه مقادیر و داده‌های پرت چهار متغیر
<span dir="ltr">Heart Rate</span>،
<span dir="ltr">Breathing Rate</span>،
<span dir="ltr">Sweating Level</span>
و
<span dir="ltr">Dizziness</span>
بررسی می‌شوند. برای متغیرهای عددی از روش
<span dir="ltr">IQR</span>
و برای متغیرهای ترتیبی و دسته‌ای از بررسی مقادیر یکتا و فراوانی استفاده می‌شود.
</p>

In [ ]:
# Create a copy and validate physiological variables

physiological_df = df[
    [
        "Heart Rate (bpm)",
        "Breathing Rate (breaths/min)",
        "Sweating Level (1-5)",
        "Dizziness"
    ]
].copy()

description = physiological_df[
    [
        "Heart Rate (bpm)",
        "Breathing Rate (breaths/min)",
        "Sweating Level (1-5)"
    ]
].describe()

display(description.T)

# Checking if numerical columns are actually numeric and with no missing values
for col in physiological_df.columns[:3]:
    physiological_df[col] = pd.to_numeric(
        physiological_df[col],
        errors="coerce"
    )

    print(
        col,
        "=>",
        physiological_df[col].isna().sum(),
        "non-numeric or missing values"
    )

print()
print(physiological_df.dtypes)

In [ ]:
# Inspect upper values and detect heart rate outliers using IQR

heart_rate = physiological_df["Heart Rate (bpm)"]

print(
    heart_rate
    .value_counts()
    .sort_index()
    .tail(10)
)

q1 = heart_rate.quantile(0.25)
q3 = heart_rate.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

heart_rate_outliers = physiological_df[
    (heart_rate < lower_bound) |
    (heart_rate > upper_bound)
]

print("\nQ1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of outliers:", len(heart_rate_outliers))

print(
    heart_rate_outliers["Heart Rate (bpm)"]
    .value_counts()
    .sort_index()
)

# Inspect related features for heart rate outliers

outlier_indices = heart_rate_outliers.index

outlier_context = df.loc[
    outlier_indices,
    [
        "Age",
        "Heart Rate (bpm)",
        "Breathing Rate (breaths/min)",
        "Sweating Level (1-5)",
        "Dizziness",
        "Stress Level (1-10)",
        "Anxiety Level (1-10)",
        "Medication"
    ]
]

display(outlier_context.head(10))

# Replace invalid heart rate values with the median

physiological_df.loc[
    physiological_df["Heart Rate (bpm)"] == 220,
    "Heart Rate (bpm)"
] = np.nan

heart_rate_median = physiological_df["Heart Rate (bpm)"].median()

physiological_df["Heart Rate (bpm)"] = (
    physiological_df["Heart Rate (bpm)"]
    .fillna(heart_rate_median)
)

print("Median used:", heart_rate_median)
print(
    "Remaining missing values:",
    physiological_df["Heart Rate (bpm)"].isna().sum()
)

In [ ]:
# Detect breathing rate outliers using IQR

breathing_rate = physiological_df["Breathing Rate (breaths/min)"]

q1_breathing = breathing_rate.quantile(0.25)
q3_breathing = breathing_rate.quantile(0.75)
iqr_breathing = q3_breathing - q1_breathing

lower_bound_breathing = q1_breathing - 1.5 * iqr_breathing
upper_bound_breathing = q3_breathing + 1.5 * iqr_breathing

breathing_outlier_mask = (
    (breathing_rate < lower_bound_breathing) |
    (breathing_rate > upper_bound_breathing)
)

breathing_rate_outliers = physiological_df[breathing_outlier_mask]

print("Q1:", q1_breathing)
print("Q3:", q3_breathing)
print("IQR:", iqr_breathing)
print("Lower bound:", lower_bound_breathing)
print("Upper bound:", upper_bound_breathing)
print("Number of outliers:", breathing_outlier_mask.sum())

print(
    breathing_rate_outliers["Breathing Rate (breaths/min)"]
    .value_counts()
    .sort_index()
)

In [ ]:
# Inspect sweating level values and frequencies

sweating_level = physiological_df["Sweating Level (1-5)"]

print(sweating_level.unique())

print(
    sweating_level
    .value_counts(dropna=False)
    .sort_index()
)

In [ ]:
# Inspect dizziness categories and frequencies

dizziness = physiological_df["Dizziness"]

print(dizziness.unique())
print(dizziness.value_counts(dropna=False))

<h4 dir="rtl" style="text-align: right;">
جمع‌بندی و مشاهدات متغیرهای فیزیولوژیک
</h4>

<p dir="rtl" style="text-align: right;">
• سه متغیر عددی با موفقیت به نوع عددی تبدیل شدند و هیچ مقدار غیرقابل‌تبدیل یا گمشده اولیه در آن‌ها مشاهده نشد.
</p>

<p dir="rtl" style="text-align: right;">
• در متغیر
<strong><span dir="ltr">Heart Rate</span></strong>
تعداد ۳۰ داده پرت شناسایی شد که همگی برابر با
<span dir="ltr">220 bpm</span>
بودند.
</p>

<p dir="rtl" style="text-align: right;">
• تکرار دقیق مقدار ۲۲۰، فاصله زیاد آن با سایر مقادیر و نبود الگوی مشخص در متغیرهای مرتبط، احتمال وجود خطای ثبت یا مقدار نامعتبر سیستماتیک را تقویت می‌کند.
</p>

<p dir="rtl" style="text-align: right;">
• در متغیر
<strong><span dir="ltr">Breathing Rate</span></strong>
هیچ داده پرتی با روش
<span dir="ltr">IQR</span>
شناسایی نشد.
</p>

<p dir="rtl" style="text-align: right;">
• متغیر
<strong><span dir="ltr">Sweating Level</span></strong>
فقط شامل مقادیر معتبر ۱ تا ۵ و متغیر
<strong><span dir="ltr">Dizziness</span></strong>
فقط شامل دسته‌های معتبر
<span dir="ltr">Yes</span>
و
<span dir="ltr">No</span>
بود.
</p>

<h4 dir="rtl" style="text-align: right;">
اقدامات انجام‌شده
</h4>

<p dir="rtl" style="text-align: right;">
• مقادیر نامعتبر ۲۲۰ در ستون
<span dir="ltr">Heart Rate</span>
ابتدا به
<span dir="ltr">NaN</span>
تبدیل و سپس با <strong>میانه مقادیر معتبر</strong> جایگزین شدند.
</p>

<p dir="rtl" style="text-align: right;">
• جایگزینی با میانه باعث حفظ تعداد ردیف‌ها و جلوگیری از ایجاد مقادیر مصنوعی خارج از توزیع اصلی شد؛ با این حال، ممکن است فراوانی داده‌ها را در اطراف میانه کمی افزایش دهد.
</p>

<p dir="rtl" style="text-align: right;">
• متغیرهای
<span dir="ltr">Breathing Rate</span>،
<span dir="ltr">Sweating Level</span>
و
<span dir="ltr">Dizziness</span>
به دلیل نداشتن مقدار نامعتبر، بدون تغییر باقی ماندند.
</p>

<h3 dir="rtl" style="text-align: right;">
3.4 درمان، سابقه و متغیر هدف:<br>

<span dir="ltr">Family History, Medication, Therapy Sessions,</span><br>

<span dir="ltr">Therapy History, Recent Major Life Event, Stress Level, Anxiety Level, Target</span>
</h3>

In [ ]:
treatment_cols = ['Family History of Anxiety', 'Medication', 'Therapy Sessions (per month)', 'Therapy History',
                  'Recent Major Life Event', 'Stress Level (1-10)', 'Anxiety Level (1-10)', 'Target', 'is_Anxious']

treatment_df = df[treatment_cols].copy()

treatment_df.describe(include='all').T

In [ ]:
# آیا Target و is_Anxious یکی هستند؟
same_target_mask = treatment_df['Target'] == treatment_df['is_Anxious']
print(f'Target == is_Anxious در {same_target_mask.sum()} سطر از {treatment_df.shape[0]}')

# Target بررسی
print()
print(treatment_df.groupby('Target')['Anxiety Level (1-10)'].agg(['min', 'max', 'count']))

stress_out_of_scale_mask = treatment_df['Stress Level (1-10)'] > 10

# استرس خارج از مقیاس
print()
print(f'استرس بالای 10: {stress_out_of_scale_mask.sum()} سطر، مقدار: {treatment_df.loc[stress_out_of_scale_mask, "Stress Level (1-10)"].unique()}')


# Therapy History
print()
print(treatment_df['Therapy History'].value_counts(dropna=False))

<h2 dir="rtl" style="text-align: right;">
    مشاهدات
</h2>

<p dir="rtl" style="text-align: right;">
    <strong>is_Anxious</strong> با <strong>Target</strong> کاملاً یکسان است و به نظر می‌رسد
    <strong>is_Anxious</strong> یک کپی از <strong>Target</strong> باشد.
</p>

<h3 dir="rtl" style="text-align: right;">
    تصمیم
</h3>

<p dir="rtl" style="text-align: right;">
    • <strong>Anxiety Level</strong> متغیر اصلی است و <strong>Target</strong> فقط برای رنگ نمودارها و آزمون‌های دسته‌ای استفاده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
    • <strong>is_Anxious</strong> به دلیل تکراری بودن حذف می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
    • <strong>Therapy History</strong> با ۱۸۲۷ مقدار گمشده از ۲۰۳۰ حذف می‌شود؛ با تنها حدود ۱۰٪ داده، نتیجه قابل اتکایی حاصل نمی‌شود.
</p>

<p dir="rtl" style="text-align: right;">
    • <strong>Stress Level = 15</strong> در ۳۰ سطر → <code>NaN</code> با ثبت فلگ → جایگزینی با <strong>میانه</strong>.
</p>

<p dir="rtl" style="text-align: right;">
    • مقادیر گمشده در <strong>Medication</strong> → <code>Unknown</code>.
</p>

In [ ]:
# فلگ و اصلاح استرس
treatment_df['stress_invalid_flag'] = stress_out_of_scale_mask.astype(int)
treatment_df.loc[stress_out_of_scale_mask, 'Stress Level (1-10)'] = np.nan

median_stress = treatment_df['Stress Level (1-10)'].median()
treatment_df['Stress Level (1-10)'] = treatment_df['Stress Level (1-10)'].fillna(median_stress)

#Medication گمشده حایگزین با Unknown
treatment_df['Medication'] = treatment_df['Medication'].fillna('Unknown')

# حذف ستون های  'is_Anxious', 'Therapy History'
treatment_df = treatment_df.drop(columns=['is_Anxious', 'Therapy History'])

print(f'میانه استرس: {median_stress}')
print(f'ستون های باقی مانده: {treatment_df.shape[1]}')

<h3 dir="rtl" style="text-align: right;">
3.5 ادغام پاک سازی ها و ساخت
<span dir="ltr">clean_df</span>
</h3>

In [ ]:
# Combine cleaned columns
clean_df = pd.concat(
    [
        demographic_df,
        lifestyle_df,
        physiological_df,
        treatment_df
    ],
    axis=1,
    verify_integrity=True
)

clean_df

<h2 dir="rtl" style="text-align: right;">
4. ویژوال تک متغیره
</h2>

<h3 dir="rtl" style="text-align: right;">
4.1 جمعیت شناختی
</h3>

<h4 dir="rtl" style="text-align: right;">
توزیع سن افراد
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Age</span>
یک متغیر عددی است. برای بررسی نحوه توزیع سن افراد از نمودار
<span dir="ltr">Histogram</span>
استفاده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
هیستوگرام مقادیر سن را در بازه‌های مختلف گروه‌بندی کرده و تعداد افراد موجود در هر بازه را نمایش می‌دهد.
با استفاده از این نمودار می‌توان شکل کلی توزیع، تمرکز داده‌ها و بازه‌های پرتکرار سن را مشاهده کرد.
</p>

In [ ]:
fig = px.histogram(demographic_df, x="Age",nbins=10, title="Age Distribution")



fig.update_layout(xaxis_title="Age", yaxis_title="Count", bargap=0.3 )

fig.show()

<h4 dir="rtl" style="text-align: right;">
نتیجه بررسی توزیع سن
</h4>

<p dir="rtl" style="text-align: right;">
نمودار توزیع سن نشان می‌دهد که افراد موجود در دیتاست در بازه تقریبی ۱۸ تا ۶۴ سال قرار دارند و داده‌ها در بخش‌های مختلف این بازه پراکنده شده‌اند.
بیشترین فراوانی در محدوده حدود ۴۰ تا ۴۵ سال مشاهده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
با توجه به اینکه در مرحله پاک‌سازی، ۶۲ مقدار گمشده ستون
<span dir="ltr">Age</span>
با میانه سن برابر با ۴۰ جایگزین شدند، بخشی از افزایش فراوانی در اطراف سن ۴۰ می‌تواند ناشی از این جایگزینی باشد.
</p>

<h4 dir="rtl" style="text-align: right;">
توزیع جنسیت
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Gender</span>
یک متغیر دسته‌ای است.
برای مقایسه فراوانی دسته‌های مختلف جنسیت از نمودار
<span dir="ltr">Bar Chart</span>
استفاده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
در نمودار میله‌ای، هر میله نمایانگر یکی از دسته‌های متغیر است و ارتفاع آن تعداد افراد موجود در آن دسته را نشان می‌دهد.
</p>

In [ ]:
gender_counts = (
    demographic_df["Gender"]
    .value_counts()
    .reset_index()
)

gender_counts.columns = ["Gender", "Count"]

gender_counts

In [ ]:
fig = px.bar(gender_counts, x="Gender", y="Count", text="Count", title="Gender Distribution")

fig.update_traces(width=0.4)

fig.update_layout(xaxis_title="Gender", yaxis_title="Count" , width=600 , height=400 )

fig.show()

<h4 dir="rtl" style="text-align: right;">
توزیع شغل افراد
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Occupation</span>
یک متغیر دسته‌ای است و شامل ۱۳ دسته شغلی مختلف می‌شود.
برای مقایسه فراوانی شغل‌ها از نمودار
<span dir="ltr">Bar Chart</span>
استفاده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
به دلیل تعداد نسبتاً زیاد دسته‌ها، نمودار به‌صورت افقی رسم می‌شود تا نام شغل‌ها خواناتر باشند و مقایسه فراوانی بین دسته‌ها ساده‌تر انجام شود.
</p>

In [ ]:
Occupation_count=(demographic_df["Occupation"].value_counts() .reset_index())
Occupation_count.columns= ['Occupation', 'Count']
Occupation_count

In [ ]:
fig = px.bar(Occupation_count,
    x="Count",
    y="Occupation",
    text="Count",
    orientation="h",
    title="Occupation Distribution")


fig.update_traces(width=0.9)


fig.update_layout( xaxis_title="Count",
    yaxis_title="Occupation")

fig.show()

<h4 dir="rtl" style="text-align: right;">
نتیجه بررسی توزیع شغل
</h4>

<p dir="rtl" style="text-align: right;">
نمودار توزیع شغل نشان می‌دهد که فراوانی دسته‌های مختلف شغلی نسبتاً نزدیک به یکدیگر است و هیچ دسته‌ای سهم غالبی از کل داده‌ها ندارد.
</p>

<p dir="rtl" style="text-align: right;">
در میان دسته‌های موجود،
<span dir="ltr">Student</span>
با ۱۸۲ نفر بیشترین فراوانی را دارد و پس از آن
<span dir="ltr">Artist</span>
با ۱۷۲ نفر و
<span dir="ltr">Lawyer</span>
با ۱۶۷ نفر قرار دارند.
در مقابل،
<span dir="ltr">Freelancer</span>
با ۱۳۴ نفر کمترین فراوانی را دارد.
</p>

<p dir="rtl" style="text-align: right;">
این نمودار تنها توزیع شغل افراد را نمایش می‌دهد و از آن نمی‌توان درباره ارتباط شغل با سطح اضطراب نتیجه‌گیری کرد.
</p>

<h3 dir="rtl" style="text-align: right;">
4.2 سبک زندگی
</h3>

<h3 dir="rtl" style="text-align: right;">
4.3 فیزیولوژیک
</h3>

<p dir="rtl" style="text-align: right;">
در این بخش، توزیع تک‌متغیره چهار ویژگی فیزیولوژیک شامل ضربان قلب، نرخ تنفس، سطح تعریق و سرگیجه بررسی می‌شود.
برای متغیرهای عددی پیوسته از هیستوگرام به همراه نمودار جعبه‌ای و برای متغیرهای ترتیبی و دسته‌ای از نمودار میله‌ای استفاده شده است.
</p>

<p dir="rtl" style="text-align: right;">
برای بررسی تأثیر مرحله پاک‌سازی بر توزیع ضربان قلب، نمودارهای قبل و بعد از جایگزینی مقادیر نامعتبر نیز با یکدیگر مقایسه می‌شوند.
</p>

In [ ]:
# Compare heart rate distributions before and after cleaning

heart_rate_before_fig = px.histogram(
    df,
    x="Heart Rate (bpm)",
    nbins=30,
    marginal="box",
    title="Distribution of Heart Rate Before Cleaning",
    color_discrete_sequence=[RED]
)

heart_rate_before_fig.update_layout(
    xaxis_title="Heart Rate (bpm)",
    yaxis_title="Number of Participants",
    bargap=0.4,
    showlegend=False,
    title_x=0.5
)

heart_rate_after_fig = px.histogram(
    physiological_df,
    x="Heart Rate (bpm)",
    nbins=30,
    marginal="box",
    title="Distribution of Heart Rate After Cleaning",
    color_discrete_sequence=[BLUE]
)

heart_rate_after_fig.update_layout(
    xaxis_title="Heart Rate (bpm)",
    yaxis_title="Number of Participants",
    bargap=0.4,
    showlegend=False,
    title_x=0.5
)

heart_rate_before_fig.show()
heart_rate_after_fig.show()

In [ ]:
# Visualize the breathing rate distribution using one-unit intervals
breathing_rate_fig = px.histogram(
    physiological_df,
    x="Breathing Rate (breaths/min)",
    nbins=18,
    marginal="box",
    title="Distribution of Breathing Rate",
    color_discrete_sequence=[BLUE]
)

breathing_rate_fig.update_traces(
    xbins=dict(
        start=11.5,
        end=29.5,
        size=1
    ),
    selector=dict(type="histogram")
)

breathing_rate_fig.update_layout(
    xaxis_title="Breathing Rate (breaths/min)",
    yaxis_title="Number of Participants",
    bargap=0.4,
    showlegend=False,
    title_x=0.5
)

breathing_rate_fig.show()

In [ ]:
# Calculate and visualize the frequency of each sweating level
sweating_counts = (
    physiological_df["Sweating Level (1-5)"]
    .value_counts()
    .sort_index()
    .rename_axis("Sweating Level (1-5)")
    .reset_index(name="Count")
)

sweating_fig = px.bar(
    sweating_counts,
    x="Sweating Level (1-5)",
    y="Count",
    text="Count",
    title="Distribution of Sweating Level",
    color_discrete_sequence=[BLUE]
)

sweating_fig.update_traces(
    textposition="outside",
    width=0.35
)

sweating_fig.update_layout(
    xaxis_title="Sweating Level",
    yaxis_title="Number of Participants",
    xaxis=dict(
        tickmode="array",
        tickvals=[1, 2, 3, 4, 5]
    ),
    showlegend=False,
    title_x=0.5
)

sweating_fig.show()

In [ ]:
# Calculate and visualize the frequency of dizziness responses
Dizziness_count = (
    physiological_df["Dizziness"]
    .value_counts()
    .reindex(["No", "Yes"])
    .rename_axis("Dizziness")
    .reset_index(name="Count")
)

Dizziness_fig = px.bar(
    Dizziness_count,
    y="Dizziness",
    x="Count",
    orientation="h",
    text="Count",
    title="Distribution of Dizziness",
    color_discrete_sequence=[BLUE]
)

Dizziness_fig.update_traces(
    textposition="outside",
    cliponaxis=False,
    width=0.4
)

Dizziness_fig.update_layout(
    xaxis_title="Number of Participants",
    yaxis_title="Dizziness",
    showlegend=False,
    title_x=0.5
)

Dizziness_fig.show()

<h4 dir="rtl" style="text-align: right;">
جمع‌بندی مشاهدات فیزیولوژیک
</h4>

<ul dir="rtl" style="text-align: right;">
    <li>
        در داده‌های اولیه ضربان قلب، تعداد ۳۰ مشاهده با مقدار
        <span dir="ltr">220 bpm</span>
        از سایر مقادیر فاصله زیادی داشتند. پس از تبدیل این مقادیر به داده گمشده و جایگزینی آن‌ها با میانه
        <span dir="ltr">92 bpm</span>،
        نقطه پرت جداشده از توزیع حذف شد.
    </li>
    <li>
        افزایش فراوانی در مقدار میانه نمودار ضربان قلب پس از پاک‌سازی، تا حدی نتیجه جایگزینی مقادیر نامعتبر با میانه است و نباید کاملاً به‌عنوان یک الگوی طبیعی تفسیر شود.
    </li>
    <li>
        نرخ تنفس در بازه
        <span dir="ltr">12–29 breaths/min</span>
        قرار دارد و مطابق بررسی انجام‌شده با روش
        <span dir="ltr">IQR</span>،
        داده پرت مشخصی در آن مشاهده نشد.
    </li>
    <li>
        سطح تعریق در هر پنج سطح مشاهده می‌شود. سطح ۴ با ۴۳۶ مشاهده بیشترین و سطح ۱ با ۳۶۰ مشاهده کمترین فراوانی را دارد؛ با این حال عدم تعادل شدیدی میان سطوح دیده نمی‌شود.
    </li>
    <li>
        متغیر سرگیجه تقریباً متعادل است؛ ۱۰۴۹ نفر معادل ۵۱٫۷ درصد پاسخ
        <span dir="ltr">Yes</span>
        و ۹۸۱ نفر معادل ۴۸٫۳ درصد پاسخ
        <span dir="ltr">No</span>
        دارند.
    </li>
</ul>

<h3 dir="rtl" style="text-align: right;">
4.4 درمان، سابقه و متغیر هدف
</h3>

<h2 dir="rtl" style="text-align: right;">
5. ویژوال دومتغیره
</h2>

<h3 dir="rtl" style="text-align: right;">
5.1 ستون های دسته ای با اضطراب
</h3>

<h3 dir="rtl" style="text-align: right;">
5.2 ستون های عددی با اضطراب
</h3>

<h2 dir="rtl" style="text-align: right;">
6. آزمون های آماری
</h2>

<h3 dir="rtl" style="text-align: right;">
6.1 آزمون همبستگی: عددی با عددی
</h3>

In [ ]:
from itertools import combinations

corr_vars = [
    "Sleep Hours",
    "Caffeine Intake (mg/day)",
    "Heart Rate (bpm)",
    "Breathing Rate (breaths/min)",
    "Physical Activity (hrs/week)"
]

corr_results = []
for col1, col2 in combinations(corr_vars, 2):
    r, p = stats.pearsonr(clean_df[col1], clean_df[col2])
    corr_results.append({
        "Variable 1": col1,
        "Variable 2": col2,
        "Pearson r": round(r, 3),
        "p-value": round(p, 5),
        "Significant (α=0.05)": p < 0.05
    })

corr_results_df = pd.DataFrame(corr_results)
corr_results_df


<h4 dir="rtl" style="text-align: right;">
6.1.2 آزمون همبستگی: عددی با عددی - ماتریس همبستگی
</h4>


In [ ]:
# Correlation matrix heatmap
corr_matrix = clean_df[corr_vars].corr()

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Correlation Matrix: Lifestyle & Physiological Variables"
)
fig.update_layout(width=700, height=600)
fig.show()

<h3 dir="rtl" style="text-align: right;">
6.2 آزمون
<span dir="ltr">t-test</span> و
<span dir="ltr">ANOVA</span>:
دسته ای با عددی
</h3>


<h4 dir="rtl" style="text-align: right;">
6.2.1 آزمون
<span dir="ltr">t-test</span>:
دسته ای با عددی
</h4>


In [ ]:
binary_vars = ["Family History of Anxiety", "Dizziness", "Smoking"]

ttest_results = []
for col in binary_vars:
    group_yes = clean_df.loc[clean_df[col] == "Yes", TARGET]
    group_no  = clean_df.loc[clean_df[col] == "No",  TARGET]

    levene_stat, levene_p = stats.levene(group_yes, group_no)
    equal_var = levene_p >= 0.05

    t_stat, t_p = stats.ttest_ind(group_yes, group_no, equal_var=equal_var)

    ttest_results.append({
        "Variable": col,
        "Mean (Yes)": round(group_yes.mean(), 2),
        "Mean (No)": round(group_no.mean(), 2),
        "Levene p": round(levene_p, 4),
        "Equal variance assumed": equal_var,
        "t-statistic": round(t_stat, 3),
        "p-value": round(t_p, 5),
        "Significant (α=0.05)": t_p < 0.05
    })

pd.DataFrame(ttest_results)



<h4 dir="rtl" style="text-align: right;">
6.2.2 آزمون
<span dir="ltr">ANOVA</span>:
دسته ای با عددی
</h4>


In [ ]:
anova_results = []

# Occupation
occupation_groups = [g[TARGET].values for _, g in clean_df.groupby("Occupation")]
levene_occ = stats.levene(*occupation_groups)
f_occ, p_occ = stats.f_oneway(*occupation_groups)

anova_results.append({
    "Variable": "Occupation",
    "Groups": len(occupation_groups),
    "Levene p": round(levene_occ.pvalue, 4),
    "F-statistic": round(f_occ, 3),
    "p-value": round(p_occ, 5),
    "Significant (α=0.05)": p_occ < 0.05
})

# Gender (excluding Unknown)
gender_df = clean_df[clean_df["Gender"] != "Unknown"]
gender_groups = [g[TARGET].values for _, g in gender_df.groupby("Gender")]
levene_gender = stats.levene(*gender_groups)
f_gender, p_gender = stats.f_oneway(*gender_groups)

anova_results.append({
    "Variable": "Gender (excl. Unknown)",
    "Groups": len(gender_groups),
    "Levene p": round(levene_gender.pvalue, 4),
    "F-statistic": round(f_gender, 3),
    "p-value": round(p_gender, 5),
    "Significant (α=0.05)": p_gender < 0.05
})

pd.DataFrame(anova_results)


<h3 dir="rtl" style="text-align: right;">
6.3 آزمون
<span dir="ltr">chi-square</span>:
دسته ای با دسته ای
</h3>

<h3 dir="rtl" style="text-align: right;">
6.4 امتیازی: بازه های اطمینان
</h3>

<h2 dir="rtl" style="text-align: right;">
7.
<span dir="ltr">KPI</span>
و فیچرهای تعاملی
</h2>

<h3 dir="rtl" style="text-align: right;">
7.1 استخراج
<span dir="ltr">KPI</span>
و فیچرهای تعاملی از ستون ها
</h3>

<h3 dir="rtl" style="text-align: right;">
7.2 ویژوال
<span dir="ltr">KPI</span>
و فیچرهای جدید
</h3>

<h2 dir="rtl" style="text-align: right;">
8. ویژوال چندمتغیره: ترکیب ویژگی ها و گروه های پرخطر
</h2>

<h2 dir="rtl" style="text-align: right;">
9. امتیازی: ویژوال سه متغیره و بیشتر
</h2>